# NCAA Bracket Model — Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def show_columns(df):
    for col in df.columns:
        print(col)

## Step 1: Build Team Season Averages

In [ ]:
season_stats = pd.read_csv("../data/raw/MRegularSeasonDetailedResults.csv")
season_stats.head()

In [ ]:
show_columns(season_stats)

In [ ]:
winners = season_stats[['Season', 'WTeamID', 'WScore', 'LScore', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF']]
winners.rename(columns={'WTeamID': 'TeamID', 'WScore': 'Score', 'LScore': 'OppScore', 'WFGM': 'FGM', 'WFGA': 'FGA', 'WFGM3': 'FGM3', 'WFGA3': 'FGA3', 'WFTM': 'FTM', 'WFTA': 'FTA', 'WAst': 'Ast', 'WTO': 'TO', 'WStl': 'Stl', 'WBlk': 'Blk', 'WPF': 'PF'}, inplace=True)
winners['Win'] = 1
winners.head()

In [ ]:
losers = season_stats[['Season', 'LTeamID', 'LScore', 'WScore', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']]
losers.rename(columns={'LTeamID': 'TeamID', 'LScore': 'Score', 'WScore': 'OppScore', 'LFGM': 'FGM', 'LFGA': 'FGA', 'LFGM3': 'FGM3', 'LFGA3': 'FGA3', 'LFTM': 'FTM', 'LFTA': 'FTA', 'LAst': 'Ast', 'LTO': 'TO', 'LStl': 'Stl', 'LBlk': 'Blk', 'LPF': 'PF'}, inplace=True)
losers['Win'] = 0
losers.head()

In [ ]:
team_stats = pd.concat([winners, losers], ignore_index=True)
team_stats.head()

In [ ]:
# Verifying the proper amount of rows
print(f"Number of rows in winners: {len(winners)}\nNumber of rows in losers: {len(losers)}\nTotal rows in team_stats: {len(team_stats)}")
if len(winners) + len(losers) == len(team_stats) and len(season_stats) * 2 == len(team_stats):
    print("The number of rows in team_stats is correct.")


In [ ]:
season_averages = team_stats.groupby(['Season', 'TeamID']).mean().reset_index().sort_values(by=['Season', 'TeamID'])    
season_averages.head()

In [ ]:
season_averages.shape

In [ ]:
checkwin = season_averages[(season_averages['Win'] < 0) | (season_averages['Win'] > 1)]
if len(checkwin) > 0:
    print("There are invalid values in the 'Win' column.")
else:
    print("All values in the 'Win' column are valid (0 or 1).")

In [499]:
print(season_averages.shape)
season_averages

(7981, 16)


,Season,TeamID,Score,OppScore,FGM,FGA,FGM3,FGA3,FTM,FTA,Ast,TO,Stl,Blk,PF,Win
0,2003,1102,57.250000,57.000000,19.142857,39.785714,7.821429,20.821429,11.142857,17.107143,13.000000,11.428571,5.964286,1.785714,18.750000,0.428571
1,2003,1103,78.777778,78.148148,27.148148,55.851852,5.444444,16.074074,19.037037,25.851852,15.222222,12.629630,7.259259,2.333333,19.851852,0.481481
2,2003,1104,69.285714,65.000000,24.035714,57.178571,6.357143,19.857143,14.857143,20.928571,12.107143,13.285714,6.607143,3.785714,18.035714,0.607143
3,2003,1105,71.769231,76.653846,24.384615,61.615385,7.576923,20.769231,15.423077,21.846154,14.538462,18.653846,9.307692,2.076923,20.230769,0.269231
4,2003,1106,63.607143,63.750000,23.428571,55.285714,6.107143,17.642857,10.642857,16.464286,11.678571,17.035714,8.357143,3.142857,18.178571,0.464286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7976,2025,1476,67.333333,70.900000,23.300000,53.966667,8.500000,24.000000,12.233333,16.966667,14.066667,11.300000,5.966667,2.600000,16.733333,0.433333
7977,2025,1477,64.354839,74.870968,23.000000,55.290323,8.387097,26.709677,9.967742,15.483871,14.161290,14.612903,8.387097,3.903226,16.774194,0.161290
7978,2025,1478,71.933333,81.400000,24.800000,55.400000,7.500000,22.900000,14.833333,20.866667,13.466667,12.933333,6.666667,2.066667,19.300000,0.233333
7979,2025,1479,65.785714,71.750000,22.678571,53.821429,7.000000,19.607143,13.428571,16.642857,13.107143,9.750000,6.607143,1.892857,16.678571,0.428571


## Step 2: Clean and Build Seed Data

In [ ]:
seeds = pd.read_csv("../data/raw/MNCAATourneySeeds.csv")
seeds.head()

In [ ]:
seed_clean = seeds.copy()
seed_clean['Seed'] = seed_clean['Seed'].str.replace(r'[a-zA-Z]', '', regex=True).astype(int)
seed_clean = seed_clean[['Season', 'TeamID', 'Seed']]
seed_clean.head()

In [ ]:
if (seed_clean['Seed'] <= 16).all() and (seed_clean['Seed'] >= 1).all():
    print("All seed values are valid (between 1 and 16).")

In [498]:
print(seed_clean.shape)
seed_clean

(2626, 3)


,Season,TeamID,Seed
0,1985,1207,1
1,1985,1210,2
2,1985,1228,3
3,1985,1260,4
4,1985,1374,5
...,...,...,...
2621,2025,1161,12
2622,2025,1213,13
2623,2025,1423,14
2624,2025,1303,15


## Step 3: Build Rankings Table (Massey)

In [ ]:
massey = pd.read_csv("../data/raw/MMasseyOrdinals.csv")
massey.head()

In [ ]:
massey_clean = massey.copy()
massey_clean = massey_clean[(massey_clean['RankingDayNum'] == 133) & massey_clean['SystemName'].isin(['POM', 'SAG', 'MOR', 'DUN'])].reset_index(drop=True)
massey_clean = massey_clean[['Season', 'TeamID', 'SystemName', 'OrdinalRank']]      
massey_clean.head()


In [ ]:
massey_pivot = massey_clean.pivot(index = ['Season', 'TeamID'], columns = 'SystemName', values = 'OrdinalRank').reset_index()
massey_pivot.columns.name = None
massey_pivot.head()

In [ ]:
massey_pivot.isnull().sum()

In [ ]:
# Cleaning up massey data
massey_pivot['DUN'] = massey_pivot.groupby('Season')['DUN'].transform(lambda x: x.fillna(x.median())) 
massey_pivot['MOR'] = massey_pivot.groupby('Season')['MOR'].transform(lambda x: x.fillna(x.median()))
massey_pivot['POM'] = massey_pivot.groupby('Season')['POM'].transform(lambda x: x.fillna(x.median()))
massey_pivot['SAG'] = massey_pivot.groupby('Season')['SAG'].transform(lambda x: x.fillna(x.median()))
massey_pivot.head() 

In [ ]:
massey_pivot.isnull().sum()

In [ ]:
massey_pivot.groupby('Season')['DUN'].count()

In [ ]:
massey_pivot.groupby('Season')['SAG'].count()

In [ ]:
#Handeling Missing Values in DUN
dun_median = massey_pivot.groupby('Season')['DUN'].median().sort_index().ffill()
print(dun_median)
season_medians = massey_pivot['Season'].map(dun_median)
massey_pivot['DUN'] = massey_pivot['DUN'].fillna(season_medians)
massey_pivot.isnull().sum()

In [ ]:
#Handeling Missing Values in SAG
sag_median = massey_pivot.groupby('Season')['SAG'].median().sort_index().ffill()
print(sag_median)
season_medians = massey_pivot['Season'].map(sag_median)
massey_pivot['SAG'] = massey_pivot['SAG'].fillna(season_medians)
massey_pivot.isnull().sum()

In [497]:
print(massey_pivot.shape)
massey_pivot

(7627, 6)


,Season,TeamID,DUN,MOR,POM,SAG
0,2003,1102,97.0,132.0,160.0,149.0
1,2003,1103,165.0,139.0,163.0,172.0
2,2003,1104,43.0,26.0,33.0,37.0
3,2003,1105,306.0,309.0,307.0,312.0
4,2003,1106,305.0,294.0,263.0,268.0
...,...,...,...,...,...,...
7622,2025,1476,299.0,328.0,322.0,182.0
7623,2025,1477,353.0,298.0,330.0,182.0
7624,2025,1478,339.0,344.0,354.0,182.0
7625,2025,1479,290.0,312.0,342.0,182.0


## Step 4: Build Tournament History Features

##### Seed Features

In [ ]:
tourney_results = pd.read_csv("../data/raw/MNCAATourneyCompactResults.csv")
tourney_results.head()

In [ ]:
bracket_winners = tourney_results[['Season', 'WTeamID', 'DayNum']]
bracket_winners.rename(columns={'WTeamID': 'TeamID'}, inplace=True)

bracket_losers = tourney_results[['Season', 'LTeamID', 'DayNum']]
bracket_losers.rename(columns={'LTeamID': 'TeamID'}, inplace=True)

In [ ]:
tourney_appearances = pd.concat([bracket_winners, bracket_losers], ignore_index=True)
tourney_appearances.head()

In [ ]:
deepest_game = tourney_appearances.groupby(['Season', 'TeamID'])['DayNum'].max().reset_index()
deepest_game

In [ ]:
conditions = [
    deepest_game['DayNum'] == 154,
    deepest_game['DayNum'] == 152,
    (deepest_game['DayNum'] >= 145) & (deepest_game['DayNum'] < 152),
    (deepest_game['DayNum'] >= 143) & (deepest_game['DayNum'] < 145),
    (deepest_game['DayNum'] >= 138) & (deepest_game['DayNum'] < 143),
    (deepest_game['DayNum'] >= 134) & (deepest_game['DayNum'] < 138),
    deepest_game['DayNum'] < 134
]

values = [6, 5, 4, 3, 2, 1, 0]

deepest_game['Round'] = np.select(conditions, values, default=0)
deepest_game

In [ ]:
# Verifying the distribution of rounds
print(deepest_game['Round'].value_counts())
print(deepest_game['Round'].max())

In [ ]:
all_teams = deepest_game['TeamID'].unique()
all_seasons = deepest_game['Season'].unique()
possible_indices = pd.MultiIndex.from_product([all_seasons, all_teams], names=['Season', 'TeamID'])
possible_indices = possible_indices.to_frame(index=False)

In [ ]:
tournament_history = pd.merge(possible_indices, deepest_game, on=['Season', 'TeamID'], how='left')
tournament_history = tournament_history.fillna({'Round': 0})
tournament_history

In [ ]:
tournament_history['round_last_year'] = tournament_history.groupby('TeamID')['Round'].shift(1)
tournament_history['round_avg_3yr'] = tournament_history.groupby('TeamID')['Round'].transform(lambda x: x.shift(1).rolling(window = 3, min_periods = 1).mean())
tournament_history['appearances_5yr'] = tournament_history.groupby('TeamID')['Round'].transform(lambda x: x.shift(1).rolling(window = 5, min_periods = 1).apply(lambda x: (x > 0).sum()))   
tournament_history 

In [ ]:
# Verifying the new features for a specific team (Duke my school & the best school OAT :))
tournament_history[tournament_history['TeamID'] == 1181]

In [ ]:
print(tournament_history.shape)
tournament_history

##### Winner Each Year Feature

In [ ]:
tourney_results.head()

In [ ]:
champions = tourney_results[['Season', 'WTeamID', 'DayNum']][tourney_results['DayNum'] == 154]
champions.rename(columns={'WTeamID': 'TeamID'}, inplace=True)
champions['won_championship_last_year'] = 1
champions['Season'] += 1
champions.head()
champions.shape

In [ ]:
champions.drop(columns=['DayNum'], inplace=True)
tournament_history = pd.merge(tournament_history, champions, on=['Season', 'TeamID'], how='left')
tournament_history = tournament_history.fillna({'won_championship_last_year': 0})
tournament_history
tournament_history.shape

In [ ]:
# Verifying the new feature for a specific team (again Duke :) )
tournament_history[tournament_history['Season'].isin([1991,1992,1993]) & (tournament_history['TeamID'] == 1181)]

In [496]:
print(tournament_history.shape)
tournament_history

(12051, 8)


,Season,TeamID,DayNum,Round,round_last_year,round_avg_3yr,appearances_5yr,won_championship_last_year
0,1985,1104,144.0,3.0,NaN,NaN,NaN,0.0
1,1985,1112,137.0,1.0,NaN,NaN,NaN,0.0
2,1985,1116,138.0,2.0,NaN,NaN,NaN,0.0
3,1985,1120,144.0,3.0,NaN,NaN,NaN,0.0
4,1985,1130,143.0,3.0,NaN,NaN,NaN,0.0
...,...,...,...,...,...,...,...,...
12046,2024,1202,NaN,0.0,2.0,0.666667,1.0,0.0
12047,2024,1244,NaN,0.0,1.0,0.333333,1.0,0.0
12048,2024,1182,138.0,2.0,0.0,0.000000,0.0,0.0
12049,2024,1212,137.0,1.0,0.0,0.000000,0.0,0.0


## Step 5: Merging All Features

In [500]:
#tournament_history = pd.merge(tournament_history, champions, on=['Season', 'TeamID'], how='left')
#tournament_history = tournament_history.fillna({'won_championship_last_year': 0})

team_features = pd.merge(tournament_history, season_averages, on=['Season', 'TeamID'], how='left')
team_features = pd.merge(team_features, seed_clean, on=['Season', 'TeamID'], how='left')
team_features = pd.merge(team_features, massey_pivot, on=['Season', 'TeamID'], how='left')

print(team_features.shape)
team_features.isnull().sum()    


(12051, 27)


Season                           0
TeamID                           0
DayNum                        9494
Round                            0
round_last_year                309
round_avg_3yr                  309
appearances_5yr                309
won_championship_last_year       0
Score                         5648
OppScore                      5648
FGM                           5648
FGA                           5648
FGM3                          5648
FGA3                          5648
FTM                           5648
FTA                           5648
Ast                           5648
TO                            5648
Stl                           5648
Blk                           5648
PF                            5648
Win                           5648
Seed                          9493
DUN                           5648
MOR                           5648
POM                           5648
SAG                           5648
dtype: int64